# 0. GARCH(1,1) for market volatility

**Goal:** forecast next month's realized variance using daily `Mkt-RF`, and compare
with last month's RV. This is forecasting stage 1 of our project, not a portfolio
backtest or a model-selection result.

We use **1963–1999** for the first fit and **2000–2009** for a development
walk-forward check. Later years are not evaluated here. These are starter
development settings, not a finalized research holdout protocol.

Extract the full starter ZIP. Open this notebook in the same directory as
`garch_baseline.py`, with `datasets/` alongside it. Install `requirements.txt`
first. All computation below calls the reusable module used by the CLI.

## 1. The model

We initially set the daily conditional mean to zero:

$$r_d=\epsilon_d,\qquad \epsilon_d=\sqrt{h_d}z_d,$$
$$h_{d+1}=\omega+\alpha\epsilon_d^2+\beta h_d.$$

- $\omega$: variance intercept (with units of daily return squared).
- $\alpha$: response to the latest squared shock.
- $\beta$: persistence of the previous conditional variance.
- $p=\alpha+\beta$: expected shock persistence in multi-step forecasts.

Gaussian QMLE uses a Gaussian likelihood as an estimation criterion; we do not
claim the actual return distribution is Gaussian. Robust standard errors are
requested. Zero mean makes the sum of predicted daily variances match our
uncentered squared-return RV target. A nonzero mean would require its squared
contribution when forecasting that same target.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from garch_baseline import (
    Config, load_data, history_at_origin, forecast_one,
    variance_path, walk_forward, score_forecasts, save_run, plot_forecasts,
)

ROOT = 'D:\\Hanyang_4(2)\\Introduction to Financial Engineering\\fe_project'
print(ROOT)
DAILY_PATH = "../../data/ff_daily_garchdata_long.parquet"
MONTHLY_PATH = "../../data/ff_monthly_baselinedata.parquet"
OUTPUT = Path('results_first_run')
OUTPUT.mkdir(exist_ok=True)
CONFIG = Config(first_forecast_month='2000-01', last_forecast_month='2025-12', horizon=21)
daily, monthly = load_data(DAILY_PATH, MONTHLY_PATH, CONFIG.series)
print(f'{len(daily):,} daily observations; {len(monthly):,} monthly observations')
print('Daily range:', daily.index.min().date(), 'to', daily.index.max().date())
print(monthly[['ret_m', 'rv', 'rv_lag1']].head().to_string())


## 2. Fit using data available at the first forecast origin

For the **January 2000** forecast, the final permissible return is **31 December
1999**. We fit in percentage units for numerical conditioning; all exported
variance values are converted to decimal-squared units by dividing by $100^2$.

The monthly official return `ret_m` is not used to estimate the daily model.
It is retained for the subsequent portfolio stage.


In [ ]:
initial_history = history_at_origin(daily, CONFIG.first_forecast_month)
initial_record, initial_fit = forecast_one(initial_history, CONFIG.horizon)
print(initial_fit.summary())
print('\nDecimal-unit forecast and diagnostics:')
print(pd.Series(initial_record).to_string())
(OUTPUT / 'initial_fit.txt').write_text(str(initial_fit.summary()) + '\n')


## 3. From daily forecasts to a monthly variance forecast

First compute $h_{D+1\mid D}$ using the latest observed shock. For later horizons:

$$h_{D+j\mid D}=\omega+(\alpha+\beta)h_{D+j-1\mid D},\quad j\geq 2.$$

Then sum the daily conditional variances:

$$\widehat V_{t+1\mid t}=\sum_{j=1}^{H}h_{D+j\mid D}.$$

Here $H=21$ is a **fixed trading-day approximation**. We do not use next month's
realized trading-day count to construct the forecast. Calendar-month RV can
cover a different number of sessions, so this is a documented approximation.
The comparison target remains actual calendar-month RV for both models.

Under zero mean this forecasts a sum of squared daily returns. It is not the
exact variance of a compounded monthly simple return. The module checks every
multi-step path against an independently coded expectation recursion.


In [ ]:
first_path = variance_path(initial_fit, CONFIG.horizon)
print(pd.DataFrame({'horizon_day': np.arange(1, CONFIG.horizon + 1),
                    'variance_decimal_squared': first_path}).head().to_string(index=False))
print('\nGARCH January forecast:', first_path.sum())
print('Lagged-RV January forecast:', monthly.loc[pd.Period('2000-01'), 'rv_lag1'])
print('January realized RV (evaluation only):', monthly.loc[pd.Period('2000-01'), 'rv'])


## 4. Walk forward through the development period

At each origin: extend the history, re-estimate parameters, generate the forecast,
and store diagnostics. Outcome columns are joined **after** all forecasts have
been constructed. The prior successful fit supplies starting values for the
next optimization; it does not lock the parameters.

The lagged-RV forecast is independently reconstructed from the prior month's
daily data and checked against `rv_lag1`. A failed fit is retried and, if still
invalid, raises an error instead of silently dropping difficult months.


In [ ]:
forecasts = walk_forward(daily, monthly, CONFIG, progress=True)
scores = score_forecasts(forecasts)
save_run(forecasts, scores, CONFIG, DAILY_PATH, MONTHLY_PATH, OUTPUT)
print('\nFirst forecast records:')
print(forecasts[['origin_date', 'n_train', 'rv_baseline_forecast',
                 'garch_rv_forecast', 'rv', 'persistence']].head().to_string())
print('\nConvergence flags:', forecasts.convergence_flag.value_counts().to_dict())
print('Fits requiring retry:', int((forecasts.fit_attempts > 1).sum()))
print('Months with recorded fit warnings:', int(forecasts.fit_warnings.ne('').sum()))


## 5. Forecast accuracy — lower is better

$$\mathrm{QLIKE}=\frac{RV}{\widehat V}-\log\left(\frac{RV}{\widehat V}\right)-1.$$

QLIKE compares relative errors and strongly penalizes forecasting too little
variance in a volatile month. MSE measures squared errors in variance levels,
so extreme months can dominate it. Both are scored on identical months.

These losses assess our RV proxy. They do not establish that either model
improves Sharpe, drawdown or net utility. No parameter tuning or variant search
was performed on these development results.


In [ ]:
print(scores.to_string(float_format=lambda x: f'{x:.8g}'))
for metric in ['QLIKE', 'MSE']:
    change = 100 * (1 - scores.loc['GARCH(1,1)', metric] / scores.loc['Lagged RV', metric])
    print(f'{metric}: GARCH loss reduction vs lagged RV = {change:.2f}%')
print('\nPositive reduction means a lower loss; negative means worse.')


In [ ]:
fig = plot_forecasts(forecasts)


## 6. What this establishes, and what comes next

The outputs establish whether this particular GARCH setup improves forecast
loss on **2000–2009 development data**, using only historical returns at each
origin. They are not final holdout evidence or a profitability finding.

Before allocation results, agree the validation/final-test split, scaling rule,
exposure cap, cost model and total-return wealth accounting. Then compare
unmanaged exposure, lagged-RV timing and GARCH timing. GJR and GINN follow as
core models, and decision-focused training follows within-model statistical
benchmarks. Identification remains optional.

Run `python -m unittest -v test_garch_baseline` for future-input perturbation,
forecast aggregation, baseline timing and loss-function checks.

**References:** [arch forecasting](https://arch.readthedocs.io/en/latest/univariate/forecasting.html),
[Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).
